In [ ]:
from dotenv import load_dotenv

from langchain_teddynote import logging
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import JinaRerank
from ast import mod

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-11")

In [ ]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [ ]:
documents = TextLoader("./data/appendix-keywords.txt").load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

texts = text_splitter.split_documents(documents)

In [ ]:
retriever = FAISS.from_documents(texts, OpenAIEmbeddings()).as_retriever(
    search_kwargs={"k": 10}
)

In [ ]:
query = "Word2Vec 에 대해서 설명해줘."

In [ ]:
docs = retriever.invoke(query)

In [ ]:
pretty_print_docs(docs)

JinaRerank 사용

In [ ]:
compressor = JinaRerank(
    model="jina-reranker-v2-base-multilingual", 
    top_n=3
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=retriever
)

In [ ]:
compressed_docs = compression_retriever.invoke("Word2Vec 에 대해서 설명해줘.")

In [ ]:
pretty_print_docs(compressed_docs)